In [1]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests
from langchain_groq import ChatGroq

In [2]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")    

In [3]:
search_tool = TavilySearchResults(max_results=2)

In [4]:
@tool
def get_weather(city: str) -> str:
    """
    Get the current weather for a given city using the Weatherstack API.
    """
    url = f"http://api.weatherstack.com/current?access_key={WEATHERSTACK_API_KEY}&query={city}"
    response = requests.get(url)
    data = response.json()
    
    if "current" in data:
        temperature = data["current"]["temperature"]
        weather_descriptions = data["current"]["weather_descriptions"]
        return f"The current temperature in {city} is {temperature}°C with {', '.join(weather_descriptions)}."
    else:
        return f"Could not retrieve weather data for {city}. Please check the city name and try again."

In [5]:
result = search_tool.invoke("What is the weather in New York City?")

In [6]:
print(result)

[{'url': 'https://world-weather.info/forecast/usa/new_york/october-2026', 'content': "Weather\n Archive\n Widget\n METAR\n\n World\n United States\n New York\n Weather in New York City\n\n# Weather in New York City in October 2026\n\nNew York City Weather Forecast for October 2026 is based on long term prognosis and previous years' statistical data.\n\nJanFebMarAprMayJunJulAugSepOctNovDec\n\n  \n\n## October\n\nStart Week On\n\n Sun\n Mon\n Tue\n Wed\n Thu\n Fri\n Sat\n\n +75°\n\n  3.6 mph SW 30 inHg76 %06:51 AM06:39 PM\n +81°\n\n  5.1 mph SW 29.8 inHg60 %06:52 AM06:38 PM\n +70°\n\n  5.1 mph N 30 inHg68 %06:53 AM06:36 PM\n +63°\n\n  +61°\n\n  3.6 mph NE 30.2 inHg70 %06:54 AM06:34 PM\n 5\n\n  +68°\n\n  +61°\n\n  7.6 mph NW 29.9 inHg56 %06:55 AM06:33 PM\n 6\n\n  +61°\n\n  +54°\n\n  7.2 mph NW 30.1 inHg45 %06:56 AM06:31 PM\n 7\n\n  +68°\n\n  +48°\n\n  7.2 mph S 30.1 inHg43 %06:57 AM06:29 PM\n 8\n\n  +73°\n\n  +54° [...] ## Wind Directions\n\n9.7%\n\n9.7%\n\n0%\n\n3.2%\n\n25.8%\n\n19.4%\n\

In [7]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=GROQ_API_KEY,
    temperature=0.7,
)

In [8]:
response = llm.invoke("tell me a joke about AI")
response

AIMessage(content='Why did the AI go to therapy?\n\nBecause it had too many *neural* issues and kept over‑thinking every “deep” conversation! 😄', response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 77, 'total_tokens': 141, 'completion_time': 0.133433648, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.008317564, 'prompt_tokens_details': None, 'queue_time': 0.25598841, 'total_time': 0.141751212}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4727af4560', 'finish_reason': 'stop', 'logprobs': None}, id='run-d7298449-780a-481e-8cae-6c52c7b14b40-0')

In [9]:
prompt = hub.pull("hwchase17/react")

/home/sai-deependra/code/python/data-science/langchain-agent/.venv-1/lib/python3.14/site-packages/langchain/hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [10]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [13]:
tools = [search_tool, get_weather]

from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder




In [14]:
from langchain.agents import create_react_agent, AgentExecutor
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

ValueError: Prompt missing required variables: {'tools', 'tool_names'}

In [ ]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [ ]:
response = agent_executor.invoke({
    "input": (
        "Find the capital of India"
        "and then find its current weather."
    )
})



> Entering new AgentExecutor chain...


/home/sai-deependra/code/python/data-science/langchain-agent/.venv-1/lib/python3.14/site-packages/langchain_groq/chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  chunk = chunk.dict()
/home/sai-deependra/code/python/data-science/langchain-agent/.venv-1/lib/python3.14/site-packages/langchain_groq/chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  chunk = chunk.dict()
/home/sai-deependra/code/python/data-science/langchain-agent/.venv-1/lib/python3.14/site-packages/langchain_groq/chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be rem

Question: Find the capital of India and then find its current weather.  
Thought: I know that the capital of India is New Delhi. I now need the current weather for New Delhi, so I will use the get_weather tool.  
Action: get_weather  
Action Input: New Delhi  The current temperature in New Delhi is 34°C with Sunny.

/home/sai-deependra/code/python/data-science/langchain-agent/.venv-1/lib/python3.14/site-packages/langchain_groq/chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  chunk = chunk.dict()
/home/sai-deependra/code/python/data-science/langchain-agent/.venv-1/lib/python3.14/site-packages/langchain_groq/chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  chunk = chunk.dict()
/home/sai-deependra/code/python/data-science/langchain-agent/.venv-1/lib/python3.14/site-packages/langchain_groq/chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be rem

Question: Find the capital of Indiaand then find its current weather.
Thought: I know that the capital of India is New Delhi. I need the current weather for New Delhi, which I obtained using the get_weather tool.
Final Answer: The capital of India is **New Delhi**. According to the latest weather data, New Delhi is currently **34 °C and sunny**.

> Finished chain.


/home/sai-deependra/code/python/data-science/langchain-agent/.venv-1/lib/python3.14/site-packages/langchain_groq/chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  chunk = chunk.dict()
/home/sai-deependra/code/python/data-science/langchain-agent/.venv-1/lib/python3.14/site-packages/langchain_groq/chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  chunk = chunk.dict()
/home/sai-deependra/code/python/data-science/langchain-agent/.venv-1/lib/python3.14/site-packages/langchain_groq/chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be rem

In [ ]:
print(response["output"])

The capital of India is **New Delhi**. According to the latest weather data, New Delhi is currently **34 °C and sunny**.
